# NutriMatch Paper-Target Prediction

This TRE notebook tries to reproduce the prediction style of the NutriMatch paper, then applies the same evaluation to other Diet Data Enhancement feature sets.

Paper-aligned idea:

1. age and sex only
2. age and sex plus basic nutrients
3. age and sex plus all NutriMatch nutrients
4. age and sex plus other enhanced diet representations

The paper highlighted body-fat indices, waist circumference, serum folate, continuous glucose monitoring traits, blood biomarkers, and 2-year overweight/obesity prediction.

In [ ]:
from pathlib import Path
import os
import sys
import re
import json
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")

RUN_TRAINING = os.environ.get("DDE_RUN_TRAINING", "0") == "1"
MODEL_NAMES = ["hist_gradient_boosting"]
CANDIDATE_MODELS_LATER = ["ridge", "random_forest", "elastic_net", "extra_trees", "lightgbm", "xgboost", "catboost"]
RANDOM_STATE = 42
MIN_N_PER_TARGET = int(os.environ.get("DDE_MIN_N_PER_TARGET", "50"))
N_SPLITS = int(os.environ.get("DDE_N_SPLITS", "5"))

print("RUN_TRAINING:", RUN_TRAINING)
print("Primary model:", MODEL_NAMES)
print("Candidate models for later, not run now:", CANDIDATE_MODELS_LATER)


## Paths And Feature Sets

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE")
if not (PROJECT_ROOT / "downstream_analysis").exists() and tre_root.exists():
    PROJECT_ROOT = tre_root

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = "nutrimatch_paper_targets_prediction"
TRE_INPUTS = PROJECT_ROOT / "tre_inputs"
CVD_OUTPUTS = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs"
RUN_ROOT = PROJECT_ROOT / "downstream_analysis/tasks" / NOTEBOOK_STEM
OUT_DIR = RUN_ROOT / "outputs"
LOG_DIR = OUT_DIR / "logs"
for p in [RUN_ROOT, OUT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Run folder:", RUN_ROOT)
print("Output directory:", OUT_DIR)
print("Background runner:", PROJECT_ROOT / "depricated/Nutrimatch_manual/run_nutrimatch_paper_targets_prediction_background.py")

FEATURE_SETS = [
    {"name": "basic_nutrimatch", "path": "outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "All NutriMatch nutrients"},
    {"name": "denovo_enriched", "path": "outputs/enhanced_hpp/1.denovo/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "De novo enriched"},
    {"name": "nutrimatch_enhanced", "path": "outputs/enhanced_hpp/3.nutrimatch_enhanced/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "NutriMatch enhanced"},
    {"name": "denovo_cardiometabolic", "path": "outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo cardiometabolic"},
    {"name": "denovo_broad_diet_health", "path": "outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo broad diet-health"},
    {"name": "denovo_microbiome", "path": "outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo microbiome-oriented"},
    {"name": "denovo_chemical_metabolomics", "path": "outputs/downstream_features/denovo/chemical_metabolomics/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo chemical/metabolomics"},
    {"name": "denovo_mental_health", "path": "outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo mental-health"},
    {"name": "nutrimatch_cardiometabolic", "path": "outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch cardiometabolic"},
    {"name": "nutrimatch_broad_diet_health", "path": "outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch broad diet-health"},
    {"name": "nutrimatch_microbiome", "path": "outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch microbiome-oriented"},
    {"name": "nutrimatch_chemical_metabolomics", "path": "outputs/downstream_features/nutrimatch_based/chemical_metabolomics/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch chemical/metabolomics"},
    {"name": "nutrimatch_mental_health", "path": "outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch mental-health"},
    {"name": "denovo_food_card_embedding", "path": "outputs/food_card/denovo/embeddings/hpp_food_card_embeddings_full_biology_text_text_embedding_3_large.parquet", "feature_mode": "embedding", "label": "Food-card embedding"},
]

FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs["path"]).exists()]
print("Feature sets found:", [fs["name"] for fs in FEATURE_SETS])


## Background Training Launcher

Run the next cell from the notebook/kernel. It starts the training script in a detached Python process, writes a PID file and log file, then returns immediately. This avoids the Jupyter `! ... &` limitation and uses the same Python environment as this kernel.


In [ ]:
# Console-only training command and status helper.
# The long model training should be launched from a terminal/console, not from the notebook browser session.
# In TRE/SageMaker, live logs are more reliable in /tmp than in the mounted project folder.

runner_path = PROJECT_ROOT / "depricated/Nutrimatch_manual/run_nutrimatch_paper_targets_prediction_background.py"
project_log_path = LOG_DIR / "background_training.log"
tmp_log_path = Path("/tmp/nutrimatch_paper_targets_prediction_background_training.log")
pid_path = LOG_DIR / "background_training.pid"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Run this from a TRE shell/terminal to create saved model results:")
print(f"cd {PROJECT_ROOT}")
print(f"mkdir -p {LOG_DIR}")
print(
    'DDE_RUN_TRAINING=1 PYTHONUNBUFFERED=1 '
    'PYTHONPATH="$PWD:${PYTHONPATH:-}" '
    f"nohup python -u {runner_path.relative_to(PROJECT_ROOT)} "
    f"> {tmp_log_path} 2>&1 &"
)
print(f"echo $! > {pid_path.relative_to(PROJECT_ROOT)}")
print(f"tail -f {tmp_log_path}")
print(f"cp {tmp_log_path} {project_log_path}")

print("\nIf you are in Python/IPython, use the Python-console launcher from depricated/Nutrimatch_manual/README.md.")
print("\nCurrent saved-artifact status:")
print("Project log exists:", project_log_path.exists(), project_log_path)
print("Tmp live log exists:", tmp_log_path.exists(), tmp_log_path)
print("PID file exists:", pid_path.exists(), pid_path)
if pid_path.exists():
    print("PID:", pid_path.read_text(encoding="utf-8").strip())
results_probe = OUT_DIR / "nutrimatch_paper_target_model_results.csv"
print("Main results exist:", results_probe.exists(), results_probe)
if results_probe.exists():
    print("Main results size:", results_probe.stat().st_size)


## Paper-Like Nutrient Arms

`paper_basic_nutrients` approximates the paper's basic HPP nutrients/macronutrients and sodium. `nutrimatch_all` uses the full NutriMatch nutrient table. Other feature sets are tested after that.

In [ ]:
PAPER_BASIC_NUTRIENT_NAMES = [
    "Energy",
    "Protein",
    "Total lipid (fat)",
    "Carbohydrate, by difference",
    "Sodium, Na",
    "Fiber, total dietary",
    "Alcohol, ethyl",
    "Water",
]

def clean_feature_name(name):
    text = str(name).replace("enriched_", "", 1).replace("kg_", "", 1).replace("food_card_", "", 1)
    return re.sub(r"\s+", " ", text).strip().lower()

BASIC_NUTRIENT_KEYS = {clean_feature_name(x) for x in PAPER_BASIC_NUTRIENT_NAMES}
print("Basic nutrient keys:", sorted(BASIC_NUTRIENT_KEYS))

## TRE Loading Helpers

In [ ]:
from downstream_analysis.data_handelling.pheno_loader_export import (
    make_loader,
    load_table_from_loader,
    dataframe_with_index_columns,
)

def read_any(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)

def find_participant_col(df):
    for col in ["participant_id", "Participant_Study_ID", "research_stage_id", "user_id", "RegistrationCode"]:
        if col in df.columns:
            return col
    for col in df.columns:
        if "participant" in str(col).lower() or "research_stage" in str(col).lower():
            return col
    return None

def try_load_pheno_table(dataset, table=None):
    try:
        loader = make_loader(dataset, age_sex_dataset=None, errors="warn")
        table_name = table or dataset
        try:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
        except Exception:
            df = None
        if df is None:
            dfs = getattr(loader, "dfs", {})
            if table_name in dfs:
                df = dataframe_with_index_columns(dfs[table_name])
            elif len(dfs) == 1:
                df = dataframe_with_index_columns(next(iter(dfs.values())))
        return df, loader
    except Exception as exc:
        print(f"Could not load {dataset}/{table or dataset}: {exc}")
        return None, None

def dataframe_brief(dataset, table, df):
    if df is None or df.empty:
        return None
    return {
        "dataset": dataset,
        "table": table,
        "rows": len(df),
        "columns": len(df.columns),
        "participant_col": find_participant_col(df),
        "numeric_columns": len(df.select_dtypes(include=np.number).columns),
    }

## Build Or Reuse Participant Diet X Tables

In [ ]:
diet_slim_csv = TRE_INPUTS / "diet_participant_food_slim.csv"
diet_events_csv = TRE_INPUTS / "diet_logging_events.csv"

if not diet_slim_csv.exists():
    if not diet_events_csv.exists():
        print("diet_logging_events.csv not found; loading diet_logging via PhenoLoader.")
        diet, _ = try_load_pheno_table("diet_logging", "diet_logging_events")
        if diet is None:
            raise FileNotFoundError("Could not load diet_logging_events from TRE.")
        TRE_INPUTS.mkdir(parents=True, exist_ok=True)
        diet.to_csv(diet_events_csv, index=False)
        del diet
        gc.collect()

    grouped_chunks = []
    for chunk in pd.read_csv(diet_events_csv, usecols=["participant_id", "food_id", "weight_g"], chunksize=500_000, low_memory=False):
        chunk["participant_id"] = chunk["participant_id"].astype(str)
        chunk["food_id"] = chunk["food_id"].astype(str)
        chunk["weight_g"] = pd.to_numeric(chunk["weight_g"], errors="coerce").fillna(0.0)
        grouped_chunks.append(chunk.groupby(["participant_id", "food_id"], as_index=False)["weight_g"].sum())

    diet_slim = pd.concat(grouped_chunks, ignore_index=True).groupby(["participant_id", "food_id"], as_index=False)["weight_g"].sum()
    diet_slim.to_csv(diet_slim_csv, index=False)
    del grouped_chunks, diet_slim
    gc.collect()

print("Using slim diet table:", diet_slim_csv)

In [ ]:
def choose_ref_food_col(ref):
    for col in ["hpp_food_id", "food_id"]:
        if col in ref.columns:
            return col
    raise ValueError("Could not find hpp_food_id or food_id in feature table")

def feature_columns(ref, ref_food_col, feature_mode):
    if feature_mode == "embedding":
        cols = [c for c in ref.columns if str(c).startswith("embedding_")]
    else:
        exclude = {ref_food_col, "food_id", "hpp_food_id", "food_name", "short_food_name", "product_name"}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    if not cols:
        numeric = ref.select_dtypes(include=[np.number]).columns.tolist()
        cols = [c for c in numeric if c != ref_food_col]
    return list(dict.fromkeys(cols))

def build_participant_x(fs, batch_size=120):
    fs_name = fs["name"]
    existing = CVD_OUTPUTS / fs_name / f"X_{fs_name}_participant.csv"
    if existing.exists():
        print("Reusing X:", existing)
        return existing

    print("Building X for", fs_name)
    ref = read_any(PROJECT_ROOT / fs["path"])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs["feature_mode"])
    ref = ref[[ref_food_col] + cols].copy()
    ref["_food_join_id"] = ref[ref_food_col].astype(str)

    diet = pd.read_csv(diet_slim_csv, low_memory=False)
    diet["participant_id"] = diet["participant_id"].astype(str)
    diet["_food_join_id"] = diet["food_id"].astype(str)
    diet["weight_g"] = pd.to_numeric(diet["weight_g"], errors="coerce").fillna(0.0)
    total_grams = diet.groupby("participant_id")["weight_g"].sum().replace(0, np.nan)

    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        merged = diet[["participant_id", "_food_join_id", "weight_g"]].merge(ref[["_food_join_id"] + batch], on="_food_join_id", how="left")
        values = merged[batch].apply(pd.to_numeric, errors="coerce").fillna(0.0)

        if fs["feature_mode"] == "enriched":
            scaled = values.mul(merged["weight_g"].to_numpy() / 100.0, axis=0)
            prefix = "enriched_"
        elif fs["feature_mode"] in ["kg", "embedding"]:
            scaled = values.mul(merged["weight_g"].to_numpy(), axis=0)
            prefix = "kg_" if fs["feature_mode"] == "kg" else "food_card_"
        else:
            raise ValueError(fs["feature_mode"])

        scaled["participant_id"] = merged["participant_id"].values
        agg = scaled.groupby("participant_id").sum(numeric_only=True)
        if fs["feature_mode"] in ["kg", "embedding"]:
            agg = agg.div(total_grams, axis=0).fillna(0.0)
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()

    x = pd.concat(parts, axis=1).reset_index()
    fs_dir = CVD_OUTPUTS / fs_name
    fs_dir.mkdir(parents=True, exist_ok=True)
    out = fs_dir / f"X_{fs_name}_participant.csv"
    x.to_csv(out, index=False)
    del ref, diet, total_grams, parts, x
    gc.collect()
    print("Wrote X:", out)
    return out

x_paths = {fs["name"]: build_participant_x(fs) for fs in FEATURE_SETS}
x_paths

## Discover Paper-Aligned Targets In TRE

This uses the official PhenoLoader dataset names from the HPP knowledgebase:

- `anthropometrics`: BMI, waist circumference, hip circumference, waist-to-hip ratio
- `body_composition`: DXA/body-fat/VAT/SAT traits
- `blood_tests`: folate, glucose, HbA1c, lipids, liver enzymes, and other labs
- `cgm`: `cgm`, `iglu`, and `iglu_daily` glucose summary tables
- `nightingale_metabolomics`: optional NMR metabolomics targets, if present in this TRE release

Age/sex/population tables are loaded separately for covariates, not as target outcomes. Review `target_catalog` after this cell. If it picks too many or too few columns, edit `SELECTED_TARGET_IDS` in the following cell.


In [ ]:
TARGET_TABLE_SPECS = [
    ("anthropometrics", "anthropometrics"),
    ("body_composition", "body_composition"),
    ("blood_tests", "blood_tests"),
    ("cgm", "cgm"),
    ("cgm", "iglu"),
    ("cgm", "iglu_daily"),
    ("nightingale_metabolomics", "nightingale_metabolomics"),
]

COVARIATE_TABLE_SPECS = [
    ("anthropometrics", "age_sex"),
    ("body_composition", "age_sex"),
    ("blood_tests", "age_sex"),
    ("cgm", "age_sex"),
    ("nightingale_metabolomics", "age_sex"),
    ("population", "population"),
]

TARGET_REGEX = re.compile(
    r"body.*fat|body_comp.*fat|fat.*percent|fat_mass|visceral|vat|sat|"
    r"waist|hip|bmi|body_mass_index|"
    r"folate|folic|b9|glucose|glyca|hba1c|hemoglobin.*a1c|"
    r"cgm|time.*range|tir|mean.*glucose|average.*glucose|"
    r"gmi|j_index|mage|conga|modd|auc|cv_glucose|sd_glucose|"
    r"obes|overweight",
    re.IGNORECASE,
)

TARGET_CATEGORY_RULES = [
    ("anthropometry", re.compile(r"waist|hip|bmi|body_mass_index", re.IGNORECASE)),
    ("body_composition", re.compile(r"body_comp|body.*fat|fat.*percent|fat_mass|visceral|vat|sat", re.IGNORECASE)),
    ("blood_glucose", re.compile(r"glucose|glyca|hba1c|hemoglobin.*a1c", re.IGNORECASE)),
    ("blood_folate", re.compile(r"folate|folic|b9", re.IGNORECASE)),
    ("cgm", re.compile(r"cgm|time.*range|tir|mean.*glucose|average.*glucose|gmi|j_index|mage|conga|modd|auc|cv_glucose|sd_glucose", re.IGNORECASE)),
    ("obesity_status", re.compile(r"obes|overweight", re.IGNORECASE)),
]

def target_category(dataset, table, column):
    text = f"{dataset} {table} {column}"
    for category, pattern in TARGET_CATEGORY_RULES:
        if pattern.search(text):
            return category
    return "other"

loaded_tables = {}
failed_table_specs = []
target_rows = []
dataset_briefs = []

for dataset, table in TARGET_TABLE_SPECS + COVARIATE_TABLE_SPECS:
    key = (dataset, table)
    if key in loaded_tables:
        continue
    frame, loader = try_load_pheno_table(dataset, table)
    if frame is None:
        failed_table_specs.append({"dataset": dataset, "table": table})
        continue
    loaded_tables[key] = frame
    brief = dataframe_brief(dataset, table, frame)
    if brief:
        dataset_briefs.append(brief)

target_keys = set(TARGET_TABLE_SPECS)
for (dataset, table), frame in loaded_tables.items():
    if (dataset, table) not in target_keys:
        continue
    pid_col = find_participant_col(frame)
    if pid_col is None:
        continue
    for col in frame.columns:
        if col == pid_col:
            continue
        if TARGET_REGEX.search(str(col)):
            numeric = pd.api.types.is_numeric_dtype(frame[col])
            if numeric or any(token in str(col).lower() for token in ["obes", "overweight"]):
                nonnull = int(frame[col].notna().sum())
                if nonnull == 0:
                    continue
                target_id = re.sub(r"[^A-Za-z0-9_]+", "_", f"{dataset}__{table}__{col}").strip("_")
                target_rows.append({
                    "target_id": target_id,
                    "category": target_category(dataset, table, col),
                    "dataset": dataset,
                    "table": table,
                    "participant_col": pid_col,
                    "column": col,
                    "numeric": bool(numeric),
                    "nonnull": nonnull,
                    "nunique": int(frame[col].nunique(dropna=True)),
                })

dataset_briefs = pd.DataFrame(dataset_briefs).drop_duplicates()
target_catalog = pd.DataFrame(target_rows).drop_duplicates("target_id")
failed_table_specs = pd.DataFrame(failed_table_specs)

dataset_briefs.to_csv(OUT_DIR / "target_dataset_briefs.csv", index=False)
target_catalog.to_csv(OUT_DIR / "paper_aligned_target_catalog.csv", index=False)
failed_table_specs.to_csv(OUT_DIR / "failed_target_table_loads.csv", index=False)

print("Loaded target/covariate tables:", len(loaded_tables))
print("Candidate targets:", len(target_catalog))
if not failed_table_specs.empty:
    print("Tables not available in this TRE release:")
    display(failed_table_specs)
display(dataset_briefs)
display(target_catalog.sort_values(["category", "dataset", "column"]).head(160))


In [ ]:
# Optional: manually choose exact target_id values from paper_aligned_target_catalog.csv.
# Leave empty to use an automatic target shortlist.
SELECTED_TARGET_IDS = []

PREFERRED_TARGET_TERMS = [
    "body_comp_total_region_percent_fat",
    "total_scan_vat_volume",
    "total_scan_vat_mass",
    "total_scan_vat_area",
    "body_comp_trunk_region_percent_fat",
    "waist_circumference",
    "waist_to_hip_ratio",
    "bmi",
    "folate",
    "folic",
    "bt__glucose_float_value",
    "bt__hba1c_float_value",
    "mean_glucose",
    "average_glucose",
    "time_in_range",
    "gmi",
    "obes",
    "overweight",
]

def target_priority(row):
    text = f"{row['dataset']} {row['table']} {row['column']}".lower()
    for i, term in enumerate(PREFERRED_TARGET_TERMS):
        if term.lower() in text:
            return i
    category_order = {
        "body_composition": 100,
        "anthropometry": 200,
        "blood_folate": 300,
        "cgm": 400,
        "blood_glucose": 500,
        "obesity_status": 600,
        "other": 999,
    }
    return category_order.get(row.get("category", "other"), 999)

if target_catalog.empty:
    raise ValueError("No paper-aligned targets discovered. Open target_dataset_briefs.csv and verify TARGET_TABLE_SPECS against the TRE data release.")

if SELECTED_TARGET_IDS:
    selected_catalog = target_catalog[target_catalog["target_id"].isin(SELECTED_TARGET_IDS)].copy()
    missing = sorted(set(SELECTED_TARGET_IDS) - set(selected_catalog["target_id"]))
    if missing:
        print("Requested target IDs not found:", missing)
else:
    selected_catalog = target_catalog.copy()
    selected_catalog["priority"] = selected_catalog.apply(target_priority, axis=1)
    selected_catalog = (
        selected_catalog
        .sort_values(["priority", "nonnull"], ascending=[True, False])
        .groupby("category", group_keys=False)
        .head(8)
        .sort_values(["priority", "nonnull"], ascending=[True, False])
        .head(40)
    )

if selected_catalog.empty:
    raise ValueError("No selected targets. Add target_id values from paper_aligned_target_catalog.csv to SELECTED_TARGET_IDS.")

selected_catalog.to_csv(OUT_DIR / "selected_paper_aligned_targets.csv", index=False)
display(selected_catalog)


## Build Participant-Level Target Table And Covariates

In [ ]:
def normalize_pid_series(s):
    return s.astype(str)

def choose_participant_target_value(frame, pid_col, col):
    keep = [pid_col, col]
    date_cols = [c for c in ["collection_date", "collection_timestamp", "date", "timestamp"] if c in frame.columns]
    part = frame[keep + date_cols].copy()
    part = part.rename(columns={pid_col: "participant_id", col: "target_value"})
    part["participant_id"] = normalize_pid_series(part["participant_id"])

    if not pd.api.types.is_numeric_dtype(part["target_value"]):
        part["target_value"] = part["target_value"].astype(str).str.lower().isin(["1", "true", "yes", "overweight", "obese", "obesity"])
        part["target_value"] = part["target_value"].astype(float)
    else:
        part["target_value"] = pd.to_numeric(part["target_value"], errors="coerce")

    part = part.dropna(subset=["target_value"])
    if part.empty:
        return pd.DataFrame(columns=["participant_id", col])

    if date_cols:
        date_col = date_cols[0]
        part[date_col] = pd.to_datetime(part[date_col], errors="coerce")
        part = part.sort_values(["participant_id", date_col])
        part = part.groupby("participant_id", as_index=False).first()
    else:
        part = part.groupby("participant_id", as_index=False)["target_value"].mean()

    return part[["participant_id", "target_value"]]

target_parts = []
for _, row in selected_catalog.iterrows():
    frame = loaded_tables[(row["dataset"], row["table"])]
    part = choose_participant_target_value(frame, row["participant_col"], row["column"])
    part = part.rename(columns={"target_value": row["target_id"]})
    target_parts.append(part)

if not target_parts:
    raise ValueError("No target tables were built. Check selected_catalog.")

targets_wide = target_parts[0]
for part in target_parts[1:]:
    targets_wide = targets_wide.merge(part, on="participant_id", how="outer")

targets_path = OUT_DIR / "paper_aligned_targets_participant.csv"
targets_wide.to_csv(targets_path, index=False)
print("Wrote targets:", targets_path, targets_wide.shape)
targets_wide.head()


In [ ]:
COVARIATE_REGEX = re.compile(r"^age$|age_at|sex$|gender$|year_of_birth", re.IGNORECASE)
covariate_frames = []

for (dataset, table), frame in loaded_tables.items():
    if (dataset, table) not in set(COVARIATE_TABLE_SPECS):
        continue
    pid_col = find_participant_col(frame)
    if pid_col is None:
        continue
    matches = [c for c in frame.columns if COVARIATE_REGEX.search(str(c))]
    if not matches:
        continue
    part = frame[[pid_col] + matches].copy().rename(columns={pid_col: "participant_id"})
    part["participant_id"] = normalize_pid_series(part["participant_id"])
    rename = {}
    for c in matches:
        lc = str(c).lower()
        if "sex" in lc or "gender" in lc:
            rename[c] = "sex"
        elif "year_of_birth" in lc:
            rename[c] = "year_of_birth"
        elif "age" in lc:
            rename[c] = "age"
    part = part.rename(columns=rename)
    keep = ["participant_id"] + [c for c in ["age", "sex", "year_of_birth"] if c in part.columns]
    part = part[keep].copy()
    if "age" in part.columns:
        part["age"] = pd.to_numeric(part["age"], errors="coerce")
    if "year_of_birth" in part.columns and "age" not in part.columns:
        part["year_of_birth"] = pd.to_numeric(part["year_of_birth"], errors="coerce")
        part["age"] = 2022 - part["year_of_birth"]
        part = part.drop(columns=["year_of_birth"])
    elif "year_of_birth" in part.columns:
        part = part.drop(columns=["year_of_birth"])
    part = part.groupby("participant_id", as_index=False).first()
    covariate_frames.append(part)

covariates = pd.DataFrame({"participant_id": targets_wide["participant_id"].astype(str).unique()})
for part in covariate_frames:
    for col in [c for c in part.columns if c != "participant_id"]:
        if col not in covariates.columns:
            covariates = covariates.merge(part[["participant_id", col]], on="participant_id", how="left")
        else:
            add = part[["participant_id", col]].rename(columns={col: f"{col}_new"})
            covariates = covariates.merge(add, on="participant_id", how="left")
            covariates[col] = covariates[col].combine_first(covariates[f"{col}_new"])
            covariates = covariates.drop(columns=[f"{col}_new"])

covariates_path = OUT_DIR / "paper_covariates_participant.csv"
covariates.to_csv(covariates_path, index=False)
print("Wrote covariates:", covariates_path, covariates.shape)
display(covariates.head())
print("Covariate non-null counts:")
display(covariates.notna().sum())


## Model Plan Note

The executable primary path now uses sklearn `HistGradientBoostingRegressor` / `HistGradientBoostingClassifier` with the old fold-by-fold scoring strategy. For regression, the reported R2 is the mean of per-fold R2 values (`r2_mean`), matching the older standalone notebook behavior that reproduced the stronger presentation-like results.

Planned later sections, not run automatically yet:

1. Exact paper-model section: reproduce the NutriMatch paper's modeling recipe as closely as the TRE data/code allow, including model class, covariates, cross-validation protocol, and metric reporting.
2. Model expansion section: compare additional candidate models such as Ridge, Random Forest, Elastic Net, Extra Trees, XGBoost/LightGBM/CatBoost where available in TRE, and calibrated classifiers for binary/quartile targets.

The previous Ridge/Random Forest run remains useful as supplementary analysis, but it is no longer the default executable model path in this notebook.


## Model Evaluation

This executable section runs the primary `HistGradientBoosting` comparison using old-style fold-by-fold scoring. Regression rows report mean fold R2, RMSE, and Pearson correlation; classification rows report mean fold AUROC plus accuracy-style metrics where defined.


In [ ]:
def model_for(task_type: str, model_name: str):
    if model_name != "hist_gradient_boosting":
        raise ValueError(f"Model is planned for later but not enabled in this run: {model_name}")
    if task_type == "classification":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)


def infer_task_type(y):
    y2 = y.dropna()
    if y2.nunique() <= 2:
        return "classification"
    return "regression"


def pipeline_for(X: pd.DataFrame, task_type: str, model_name: str) -> Pipeline:
    numeric_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]
    pre = ColumnTransformer(
        transformers=[
            # SimpleImputer() defaults to mean imputation, matching the old notebook path.
            ("num", Pipeline([("impute", SimpleImputer()), ("scale", StandardScaler())]), numeric_cols),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop",
    )
    return Pipeline([("pre", pre), ("model", model_for(task_type, model_name))])


def usable_cv(y: pd.Series, task_type: str, n_splits: int = N_SPLITS):
    if task_type == "classification":
        counts = y.value_counts(dropna=True)
        if counts.empty or counts.min() < 2:
            return None
        k = min(n_splits, int(counts.min()))
        if k < 2:
            return None
        return StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    return KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


def evaluate_arm(X, y, *, task_type, model_name, n_splits=N_SPLITS):
    valid = y.notna()
    X = X.loc[valid].copy()
    y = y.loc[valid].copy()
    if len(y) < MIN_N_PER_TARGET:
        return None

    class_count = None
    if task_type == "classification":
        y = y.astype("Int64").astype(str) if pd.api.types.is_numeric_dtype(y) else y.astype(str)
        class_count = int(y.nunique())
        if class_count < 2 or y.value_counts().min() < 2:
            return None

    cv = usable_cv(y, task_type, n_splits=n_splits)
    if cv is None:
        return None

    estimator = pipeline_for(X, task_type, model_name)
    split_iter = cv.split(X, y) if task_type == "classification" else cv.split(X)
    rows = []

    for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
        est = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        est.fit(X_train, y_train)
        pred = est.predict(X_test)

        if task_type == "classification":
            row = {
                "fold": fold,
                "accuracy": float(accuracy_score(y_test, pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
                "f1_macro": float(f1_score(y_test, pred, average="macro", zero_division=0)),
                "auc": np.nan,
                "auprc": np.nan,
            }
            try:
                if hasattr(est, "predict_proba"):
                    score = est.predict_proba(X_test)[:, -1]
                else:
                    score = est.decision_function(X_test)
                labels = sorted(pd.Series(y_test).dropna().unique())
                positive = labels[-1]
                row["auc"] = float(roc_auc_score(y_test, score))
                row["auprc"] = float(average_precision_score((pd.Series(y_test).to_numpy() == positive).astype(int), score))
            except Exception:
                pass
            rows.append(row)
        else:
            try:
                pr = float(pearsonr(y_test, pred).statistic)
            except Exception:
                pr = np.nan
            rows.append({
                "fold": fold,
                "r2": float(r2_score(y_test, pred)),
                "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
                "pearson_r": pr,
            })

    fold_df = pd.DataFrame(rows)
    task_label = f"classification ({class_count} classes)" if task_type == "classification" else "regression"
    out = {
        "n": int(len(y)),
        "feature_count": int(X.shape[1]),
        "task": task_label,
        "task_type": task_type,
        "class_count": class_count,
        "model": model_name,
    }
    for col in fold_df.columns:
        if col == "fold":
            continue
        out[f"{col}_mean"] = float(fold_df[col].mean())
        out[f"{col}_std"] = float(fold_df[col].std())
    return out


In [ ]:
def load_x(feature_set):
    x = pd.read_csv(x_paths[feature_set], low_memory=False)
    x["participant_id"] = x["participant_id"].astype(str)
    if "time_window" in x.columns:
        x = x.drop(columns=["time_window"])
    return x

basic_x = load_x("basic_nutrimatch")
all_basic_feature_cols = [c for c in basic_x.columns if c != "participant_id" and pd.api.types.is_numeric_dtype(basic_x[c])]
paper_basic_cols = [c for c in all_basic_feature_cols if clean_feature_name(c) in BASIC_NUTRIENT_KEYS]
print("Paper-basic nutrient columns found:", paper_basic_cols)

if not paper_basic_cols:
    raise ValueError("No paper-basic nutrient columns were found in basic_nutrimatch X.")

In [ ]:
covariate_cols = [c for c in ["age", "sex"] if c in covariates.columns and covariates[c].notna().any()]
print("Using covariates:", covariate_cols)

arms = [
    {"arm": "age_sex_only", "feature_set": "covariates", "label": "Age + sex only", "x": covariates[["participant_id"] + covariate_cols].copy()},
    {"arm": "paper_basic_nutrients", "feature_set": "basic_nutrimatch", "label": "Age + sex + basic nutrients", "x": basic_x[["participant_id"] + paper_basic_cols].merge(covariates[["participant_id"] + covariate_cols], on="participant_id", how="left")},
    {"arm": "nutrimatch_all", "feature_set": "basic_nutrimatch", "label": "Age + sex + all NutriMatch nutrients", "x": basic_x.merge(covariates[["participant_id"] + covariate_cols], on="participant_id", how="left")},
]

for fs in FEATURE_SETS:
    if fs["name"] == "basic_nutrimatch":
        continue
    x = load_x(fs["name"]).merge(covariates[["participant_id"] + covariate_cols], on="participant_id", how="left")
    arms.append({"arm": fs["name"], "feature_set": fs["name"], "label": "Age + sex + " + fs["label"], "x": x})

print("Arms:")
for arm in arms:
    print(arm["arm"], arm["x"].shape)

In [ ]:
results_path = OUT_DIR / "nutrimatch_paper_target_model_results.csv"
ridge_rf_archive_path = OUT_DIR / "supplementary_ridge_random_forest_model_results.csv"

if RUN_TRAINING:
    if results_path.exists() and not ridge_rf_archive_path.exists():
        try:
            previous_results = pd.read_csv(results_path, low_memory=False)
            previous_models = set(previous_results.get("model", pd.Series(dtype=str)).dropna().astype(str))
            if previous_models.intersection({"ridge", "random_forest"}):
                previous_results.to_csv(ridge_rf_archive_path, index=False)
                print("Archived previous Ridge/Random Forest supplementary results:", ridge_rf_archive_path)
        except Exception as exc:
            print("Could not archive previous results before training:", exc)

    results = []

    for target_id in selected_catalog["target_id"]:
        y_table = targets_wide[["participant_id", target_id]].copy()
        y_table["participant_id"] = y_table["participant_id"].astype(str)
        y = y_table[target_id]
        task_type = infer_task_type(y)
        target_meta = selected_catalog[selected_catalog["target_id"].eq(target_id)].iloc[0].to_dict()
        print("Target:", target_id, "task:", task_type)

        for arm in arms:
            merged = arm["x"].merge(y_table, on="participant_id", how="inner")
            X = merged.drop(columns=["participant_id", target_id])
            y_aligned = merged[target_id]
            for model_name in MODEL_NAMES:
                metrics = evaluate_arm(X, y_aligned, task_type=task_type, model_name=model_name, n_splits=N_SPLITS)
                if metrics is None:
                    print("  skipped", arm["arm"], model_name)
                    continue
                row = {
                    "target_id": target_id,
                    "target_column": target_meta["column"],
                    "target_dataset": target_meta["dataset"],
                    "target_table": target_meta["table"],
                    "arm": arm["arm"],
                    "feature_set": arm["feature_set"],
                    "label": arm["label"],
                    **metrics,
                }
                results.append(row)
                key_metric = row.get("auc_mean") if task_type == "classification" else row.get("r2_mean")
                print(" ", arm["arm"], model_name, key_metric)

    results = pd.DataFrame(results)
    results.to_csv(results_path, index=False)
    print("Wrote:", results_path)
elif results_path.exists():
    results = pd.read_csv(results_path, low_memory=False)
    print("Loaded saved results:", results_path, results.shape)
else:
    results = pd.DataFrame()
    print("Training skipped and no saved results found yet.")
    print("Run the console command above, wait for completion, then rerun this notebook to load results.")

display(results.head() if not results.empty else results)


## Deltas Against Paper-Like Baselines

In [ ]:
def normalize_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    legacy = {
        "r2_mean": "r2",
        "rmse_mean": "rmse",
        "pearson_r_mean": "pearson_r",
        "auc_mean": "auroc",
        "auprc_mean": "auprc",
        "accuracy_mean": "accuracy",
        "balanced_accuracy_mean": "balanced_accuracy",
        "f1_macro_mean": "f1_macro",
    }
    for old, new in legacy.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]
    if "task" not in df.columns and "task_type" in df.columns:
        df["task"] = df["task_type"]
    if "model" not in df.columns:
        df["model"] = "model"
    if "class_count" not in df.columns:
        df["class_count"] = np.nan
    if "primary_metric_name" not in df.columns:
        df["primary_metric_name"] = np.where(df.get("task_type", "").eq("regression"), "R2", "AUROC")
    if "primary_metric" not in df.columns:
        df["primary_metric"] = np.where(df.get("task_type", "").eq("regression"), df.get("r2"), df.get("auroc"))
    return df


def add_baseline_delta(results: pd.DataFrame, baseline_arm: str, metric: str) -> pd.DataFrame:
    base = results[results["arm"].eq(baseline_arm)][["target_id", "model", metric]].rename(columns={metric: f"{baseline_arm}_{metric}"})
    out = results.merge(base, on=["target_id", "model"], how="left")
    out[f"delta_{metric}_vs_{baseline_arm}"] = out[metric] - out[f"{baseline_arm}_{metric}"]
    return out


if results.empty or "task_type" not in results.columns:
    print("No model results available yet; skipping deltas until the background run finishes.")
    results_for_tables = pd.DataFrame()
    metric_table = pd.DataFrame()
    best_by_target = pd.DataFrame()
    arm_summary = pd.DataFrame()
    reg = pd.DataFrame()
    clf = pd.DataFrame()
    reg_delta = pd.DataFrame()
    clf_delta = pd.DataFrame()
else:
    results = normalize_metric_columns(results)
    results["primary_metric_name"] = np.where(results["task_type"].eq("regression"), "R2", "AUROC")
    results["primary_metric"] = np.where(results["task_type"].eq("regression"), results["r2"], results["auroc"])

    reg = results[results["task_type"].eq("regression")].copy()
    clf = results[results["task_type"].eq("classification")].copy()

    reg_delta = add_baseline_delta(reg, "paper_basic_nutrients", "r2") if not reg.empty else pd.DataFrame()
    reg_delta = add_baseline_delta(reg_delta, "nutrimatch_all", "r2") if not reg_delta.empty else reg_delta
    clf_delta = add_baseline_delta(clf, "paper_basic_nutrients", "auroc") if not clf.empty else pd.DataFrame()
    clf_delta = add_baseline_delta(clf_delta, "nutrimatch_all", "auroc") if not clf_delta.empty else clf_delta

    if not reg_delta.empty:
        reg_delta["delta_vs_nutrimatch_all"] = reg_delta["delta_r2_vs_nutrimatch_all"]
    if not clf_delta.empty:
        clf_delta["delta_vs_nutrimatch_all"] = clf_delta["delta_auroc_vs_nutrimatch_all"]
    results_for_tables = pd.concat([reg_delta, clf_delta], ignore_index=True, sort=False)

    reg_delta.to_csv(OUT_DIR / "regression_deltas_vs_paper_baselines.csv", index=False)
    clf_delta.to_csv(OUT_DIR / "classification_deltas_vs_paper_baselines.csv", index=False)

    metric_cols = [
        "target_dataset", "target_table", "target_column", "task", "model", "label",
        "n", "feature_count", "r2_mean", "r2", "rmse_mean", "rmse", "pearson_r_mean", "pearson_r",
        "accuracy_mean", "accuracy", "balanced_accuracy_mean", "balanced_accuracy", "f1_macro_mean", "f1_macro", "auc_mean", "auroc", "auprc_mean", "auprc",
        "delta_r2_vs_paper_basic_nutrients", "delta_r2_vs_nutrimatch_all",
        "delta_auroc_vs_paper_basic_nutrients", "delta_auroc_vs_nutrimatch_all",
    ]
    metric_cols = [c for c in metric_cols if c in results_for_tables.columns]
    metric_table = results_for_tables[metric_cols].copy()
    metric_table.to_csv(OUT_DIR / "paper_target_metric_table.csv", index=False)
    print("Wrote:", OUT_DIR / "paper_target_metric_table.csv")

    best_by_target = (
        results_for_tables.dropna(subset=["primary_metric"])
        .sort_values("primary_metric", ascending=False)
        .groupby(["target_id", "model", "task_type"], as_index=False)
        .head(1)
    )
    best_by_target.to_csv(OUT_DIR / "paper_target_best_by_target.csv", index=False)
    print("Wrote:", OUT_DIR / "paper_target_best_by_target.csv")

    arm_summary = results.groupby(["arm", "label", "model", "task", "task_type"], as_index=False).agg(
        targets=("target_id", "nunique"),
        mean_r2=("r2", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_auroc=("auroc", "mean"),
        mean_auprc=("auprc", "mean"),
        mean_primary_metric=("primary_metric", "mean"),
    )
    arm_summary.to_csv(OUT_DIR / "paper_target_arm_summary.csv", index=False)
    print("Wrote:", OUT_DIR / "paper_target_arm_summary.csv")

    print("\nMetric table: which data gives what accuracy for each task")
    display(metric_table.sort_values(["target_dataset", "target_column", "model", "label"]).head(200))
    print("\nBest data source per target/model")
    display(best_by_target[[c for c in ["target_dataset", "target_column", "task", "model", "label", "primary_metric_name", "primary_metric", "n", "feature_count"] if c in best_by_target.columns]].head(100))
    print("\nArm summary")
    display(arm_summary.sort_values(["task_type", "mean_primary_metric"], ascending=[True, False], na_position="last"))


## Plots

In [ ]:
if not reg.empty:
    plot = reg.copy()
    target_order = plot.groupby("target_column")["r2"].max().sort_values(ascending=False).index
    keep_targets = list(target_order[:20])
    plot = plot[plot["target_column"].isin(keep_targets)]

    fig, ax = plt.subplots(figsize=(14, max(6, 0.35 * len(keep_targets))))
    sns.barplot(data=plot, x="r2", y="target_column", hue="label", order=keep_targets, ax=ax)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("Paper-aligned regression targets: R2 by model arm")
    ax.set_xlabel("5-fold CV R2")
    ax.set_ylabel("Target")
    ax.legend(title="Model arm", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    path = OUT_DIR / "paper_aligned_regression_r2_by_arm.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)


In [ ]:
if not reg_delta.empty:
    enhanced_only = reg_delta[~reg_delta["arm"].isin(["age_sex_only", "paper_basic_nutrients", "nutrimatch_all"])].copy()
    heat = enhanced_only.pivot_table(index="label", columns="target_column", values="delta_r2_vs_nutrimatch_all", aggfunc="mean")
    row_order = enhanced_only.groupby("label")["delta_r2_vs_nutrimatch_all"].mean().sort_values(ascending=False).index
    col_order = enhanced_only.groupby("target_column")["delta_r2_vs_nutrimatch_all"].max().sort_values(ascending=False).index
    heat = heat.reindex(index=row_order, columns=col_order)

    fig, ax = plt.subplots(figsize=(15, max(6, 0.45 * len(heat))))
    sns.heatmap(heat, center=0, cmap="RdBu_r", linewidths=0.4, linecolor="white", cbar_kws={"label": "Delta R2 vs all NutriMatch nutrients"}, ax=ax)
    ax.set_title("Other enhanced data vs all NutriMatch nutrients")
    ax.set_xlabel("Target")
    ax.set_ylabel("")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    path = OUT_DIR / "enhanced_vs_nutrimatch_all_delta_r2_heatmap.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)


In [ ]:
if not reg_delta.empty:
    best = (
        reg_delta[~reg_delta["arm"].isin(["age_sex_only", "paper_basic_nutrients", "nutrimatch_all"])]
        .sort_values("delta_r2_vs_nutrimatch_all", ascending=False)
        .head(20)
        .copy()
    )
    best["example"] = best["target_column"].astype(str) + " | " + best["label"].astype(str) + " | " + best["model"].astype(str)
    best.to_csv(OUT_DIR / "best_other_enhanced_vs_nutrimatch_all.csv", index=False)

    fig, ax = plt.subplots(figsize=(12, max(5, 0.45 * len(best))))
    sns.barplot(data=best.sort_values("delta_r2_vs_nutrimatch_all"), x="delta_r2_vs_nutrimatch_all", y="example", ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Best examples: other enhanced data beating all NutriMatch nutrients")
    ax.set_xlabel("Delta R2 vs all NutriMatch nutrients")
    ax.set_ylabel("")
    plt.tight_layout()
    path = OUT_DIR / "best_other_enhanced_vs_nutrimatch_all.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)
    display(best[["target_column", "task", "model", "label", "r2", "nutrimatch_all_r2", "delta_r2_vs_nutrimatch_all", "n", "feature_count"]])


In [ ]:
# Supplement-ready summary table. The detailed per-target table is written in the delta cell above.
if "arm_summary" not in globals() or arm_summary.empty:
    summary = pd.DataFrame()
    print("No summary available yet; run background training first.")
    display(summary)
else:
    summary = arm_summary.copy()
    display(summary.sort_values(["task_type", "mean_primary_metric"], ascending=[True, False], na_position="last"))
